In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import numpy as np
import pandas as pd

sys.path.append('./src')
import tl_field_selector as fs
import tl_cat_creator as tlc
import visualisation as vs
import optimize_fields as opt
import tl_guide_fibre_allocator as gfa


from astropy.io import fits
from astropy.table import Table

from tl_protofielding import TLDriverCat, create_plate_fits_file

# Import after the previous packages to enable correct plotting
%matplotlib inline
import matplotlib.pyplot as plt

# Warnings regarding stage6 can be safely ignored at the time of writing. 

# DEFINING WTL (WEAVE-Tumble-Less) FIELDS

 ## Before you start:
 
**DISCLAIMER: This mode is currently under development and may produce unexpected results!** </br> </br>

**IMPORTANT** </br>
**1. To run this notebook correctly your IP has to be whitelisted on WASP!** (wasp.ast.cam.ac.uk) </br>

**2. If you do not have a WASP validated FITS file yet, you can use a different target identifier in your fits file (src_id_col).** </br>
**However, the FITS file must have been created in a WASP-compliant format. This means that all its mandatory columns, except CNAME, have to be filled!**

**3. It is important that you run the OB creation for every field plate SEPARATELY!** </br> </br>


**General Info:** </br>
**"WTL"** stands for **"WEAVE Tumble-Less"**, and refers to the fact that it enables observations of multiple fields without the need to tumble the field plates. This means that you can **observe up to 3 fields (pointings) with a single fibre configuration**. A **collection of fields** that are observed with the same fibre configuration is referred to as a **"field batch"**. </br>
This observing mode obviously places strict limits on the number of targets that can be observed for each of the individual field. </br>
It is therefore necessary that you define your input catalogues accordingly. </br> </br>
**This means that the target density per 2 square degrees (WEAVE FOV) should not exceed 300, as there are only ~900 fibres on each plate!** </br>
The routine below is **optimised to maximize the number of fields per batch, prioritizing batches with 3 fields**. </br>
The on-sky offset between individual fields has been succesfully tested to up to 15 deg. However the real **limit is set by atmospheric differential refraction**. </br>
For good measure we recommend **offsets between fields should not be larger than 5 deg!** </br>
If you have **offsets between target groups** in your input catalogue that are **larger than 5 deg** you should consider **splitting your input catalogue** into different regions that have coherent coordinate coverage. </br> 
This can be done easily by **modfying the pandas.DataFrame** after reading in the fits catalogue. Then the notebook can be **run for each of the dataframes**.


## Open the WASP FITS file and extract its contents as a pandas.DataFrame

In [ ]:
fits_cat_path = "WS2025XX-XXX.fits"
targsrvy = os.path.splitext(os.path.basename(fits_cat_path))[0]
src_id_col = 'CNAME'
print(targsrvy)


In [ ]:
with fits.open(fits_cat_path) as hdu:
    cat_df = Table(hdu[1].data).to_pandas()

print(cat_df.shape)


In [ ]:
vs.plot_input_fits_cat(cat_df, center=(242.5, 53))

## Add a Healpix ID column to enable WEAVE Guide Catalogue (WG) query

In [ ]:
cat_df = tlc.add_healpix_id(cat_df, ra_col='GAIA_RA', de_col='GAIA_DEC',
                            hp_col='healpixId', nside=32, nested=True)

# Get the healpix ids for querying the WEAVE guide star catalogue
healpix_ids = cat_df['healpixId'].unique()
print(healpix_ids)

## Query WEAVE Guide Catalogue (This can take time, depending on the how many healpix ids you are querying)!
**Also make sure that your IP-adress is whitelisted on WASP!**

In [ ]:
tlc.query_weave_guide_stars(healpix_ids=healpix_ids,
                        split_nr=None,
                        mag_low=14.5,
                        mag_high=15,
                        cache=False,
                        neighbour_hp=True,
                        out_dir='./resources/WG')

## Read-in the created WG parquet file and combine it with the input catalogue
The default name for the combined WG parquet is **"WG_all.parquet"**. </br>
Depending on the size and on-sky extension of targets in your input catalogue, this step can produce very large files. For highly extended target lists we recommend splitting the catalogue intp subsets before defining the TL-fields.

In [ ]:
wg_df = pd.read_parquet('./resources/WG/WG__all.parquet')
cat_comb_df = tlc.combine_wg_with_input_cat(cat_df, wg_df, src_id_col=src_id_col)
#print(cat_comb_df.sample(10))

## Define on-sky fields
The function below enables the creation of WTL field configurations. <br>
**Returns a dataframe cotaining all targets that are assigned to fields as well as their centre guide stars.** </br>
The second dataframe returned is a lookup table tracking field assignments. This is used as input for the guide fibre allocation. </br>

There are multiple ways of defining WTL fields. <br> There is no one correct way of doing things, and the chosen method mostly depends on the on-sky coordinate distribution of the input catalogue. <br>
For **fields with a small on-sky extent**, it is **often easier to provide custom field centres** than for large input catalogues that extend over tens of degrees.


1. Fixed grid method: <br>
   You can start with a fixed grid of circles spanning the coordinate range of the input catalogue **(fixed_grid=True)**. <br>
   The offset between circle centres is defined by the **grid_offset** (deg) parameter. <br>
   The centre points of these circles will snap to the nearest available guide star. <br>
   It is highly **recommended to set prune_fields=True** when using this method.

2. Custom field centres: <br>
   A pandas Dataframe containing coordinates of desired centre points (see below) can be supplied as the **circles_df** parameter. <br>
   In this case the function finds the closest suitable guide star to each of the custom coordinates and sets it as a field centre. <br>
   This method automatically **ignores the fixed_grid and prune_fields parameters**.

3. Automatic (greedy) field definition: <br>
   This approach offers a heuristic solution to the geometric set cover problem over the input coordinate space. <br>
   It considers only guide stars as field centres and tries to maximize science target coverage while minimizing the needed number of fields. <br>
   To activate you have to set **fixed_grid=False**. Here, **prune_fields can be set to True or False**. <br>
   However, it will sometimes (but not exclusively) give better results when set to True. <br>

**If the above methods do not produce satisfactory results, try splitting the input catalogue into different subsets and running the process again on each one. This may lead to better outcomes.** <br>
For the example in this notebook, the custom field centres method produces good results, but when compared with the automatic approach it is missing 2 science targets.



In [ ]:
custom_field_centres = pd.DataFrame({'GAIA_RA':[241.5, 244.1, 242.8, 242.73],
                                     'GAIA_DEC':[55.0, 55.1, 55.9, 53.75]})

In [ ]:
pd.set_option('display.max_rows', None)
field_stars, lut_df = fs.define_fields(cat_comb_df, grid_offset=0.61,
                                        central_star_mag_lim=16.,
                                        src_id_col=src_id_col,
                                        verbose=True,
                                        fixed_grid=False, # Set False for automatic field definition. 
                                        circles_df=custom_field_centres,
                                        prune_fields=True)


vs.plot_tl_fov_coverage(field_stars_df=field_stars, center=(242.5, 53),
                     coords='radec', figsize=(7, 6))



## Experimental:
We want to optimize the search for a good grid_offset parameter. This parameter defines the offsets in degree on the sky of a uniformly spaced grid of 2 deg circles (FOV) on the sky. In define_fields() it is used as an intial configuration in the optimization of best field coverage, using a minimal amount of circles. The cell below runs the function with different grid_offset values over a given range. The default range runs from 0.3 to 1.3 in steps of 0.01. If the default option is not satisfactory, you can provide your own offset range. 

In [ ]:
best_grid_offset = opt.sweep_offsets(cat_comb_df,
                                fs,
                                offsets=np.round(np.arange(0.6, 1.2 + 1e-9, 0.01), 3), # supply custom offset range here
                                central_star_mag_lim=16.,
                                src_id_col=src_id_col,
                                fixed_grid=True,
                                prune_fields=True)

print("Best grid_offset:", best_grid_offset["offset"])
print("Metrics:", best_grid_offset["metrics"])

## Create a dictionary containing the fibre ids of guide fibres NOT in operation, for both field plates
Here we use the fibre status file. It can be downloaded from confluence. In the future we are planning to do this automatically in the background.

In [ ]:
fibre_file_path = './resources/fibresINGGenerated_20260903.xml'


# To check for avaiable guide fibres set fibre_ids_to_check=None!
broken_fibres = gfa.check_guide_fibre_status(fibre_status_xml=fibre_file_path, fibre_ids_to_check=None)
print(broken_fibres)

## Allocate suitable guide fibres for each merged field configuration. </br></br> **This only has to be done for each plate. The PA orientation is determined automatically for each batch from the field-centre declinations.**
Disabled or broken guide fibres are taken into account.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

plate='A' # Set plate accordingly

coverage = gfa.run_guide_fibre_allocation(field_stars,
                                          cat_comb_df,
                                          lut_df,
                                          plate=plate,
                                          guide_mag_limit=16.0,
                                          verbose=True,
                                          src_id_col=src_id_col,
                                          broken_fibres=broken_fibres,
                                          add_sky_targets=True,
                                          max_num_sky_targets=15,
                                          outpath='./')




# OB-XML CREATION

## Define necessary parameters for the PHU of the plate-FITS files. The plate-FITS files are only used to create the TL-OB-xmls and should NOT be uploaded to WASP! 

In [ ]:
print(f'./{targsrvy}_{plate}_AUTO.fits')


In [ ]:
progtemp, obstemp = create_plate_fits_file(infile=fits_cat_path,
                                           coverage_df=coverage, plate=plate,
                                           src_id_col=src_id_col,
                                           outpath='./')

# Check if your PROGTEMP and OBSTEMP are correct!
print("PROGTEMP:", progtemp)
print("OBSTEMP: ", obstemp)


## Create TL-Driver catalogue object. This enables the creation of TL-OB-xmls. 

In [ ]:
tl_driver = TLDriverCat(filename=f'./WS2025XX-XXX_{plate}_AUTO.fits',
                        guide_star_csv=f'./guide_stars_{plate}_AUTO.csv',
                        expcal_json='./resources/expcal.json')




## Also we define the individual field batches by extracting the unique batch ids from the guide stars CSV created when running create_plate_fits_file().

In [ ]:
target_csv_path = f'./central_target_list_{plate}_AUTO.csv'
central_target_df = pd.read_csv(target_csv_path)
print(central_target_df.shape[0], 'central guide stars found!')
field_batch_ids = central_target_df['batchId'].unique()


## Create the TL-CONFIGURE ready OB xmls
Adding calibration stars (calib_stars=True) ONLY works for targets with CNAMEs (created from a WASP validated FITS file)!!! </br>
When adding calibration stars you might see some Warnings regarding **"missing keys in the target attribute lookup"**. </br> You can safely ignore them. </br>
The same is true for warnings regarding the **"obsgroup"** functionality, which is not yet implemented.

In [ ]:
for batch_id in field_batch_ids:
    tl_driver.process_tl_ob(batch_id, target_csv_path,
                              progtemp=progtemp,
                              obstemp=obstemp,
                              plate=f'PLATE_{plate}',
                              calib_stars=False,
                              output_dir=f'./xmls/plate_{plate}/',
                              guide_mag_lim=16.,
                              calib_mag_lim=20,#17.9,
                              min_guide_cut=0.1,
                              max_guide_cut=0.98,
                              suffix=batch_id,
                              verbose=False,
                              ob_class='science',
                              field_name='DEMO')

print('--------------------------------- DONE --------------------------------')


## Set the epoch for TL-CONFIGURE

Jan 1st: 2025.000 <br>
Feb 1st: 2025.083 <br>
Mar 1st: 2025.166 <br>
Apr 1st: 2025.250 <br>
May 1st: 2025.333 <br>
Jun 1st: 2025.416 <br>
Jul 1st: 2025.500 <br>
Aug 1st: 2025.583 <br>
Sep 1st: 2025.666 <br>
Oct 1st: 2025.750 <br>
Nov 1st: 2025.833 <br>
Dec 1st: 2025.916 <br>


epoch = year + (month-1)/12

In [ ]:
epoch=2025.750

## Run TL-CONFIGURE. The most recent fibre status XML should be supplied in --fibre_status!!! We already defined its path above, so we use fibre_file_path again. 

In [ ]:
print(fibre_file_path)
print(os.path.isfile(fibre_file_path))

In [ ]:
! ./configure/bash_scripts/./run_tl_configure.sh --epoch {epoch} --plate {plate} --fibre_status {fibre_file_path}

## Define paths for final TL-CONFIGURE output

In [ ]:
configure_output_path = f'./xmls/configure_{plate}/'
print('Finalization input path:', configure_output_path)
final_configure_output_path = f'./xmls/configure_{plate}/finalized_{plate}/' #{trimester}/'
print('Finalization output path:', final_configure_output_path)

## Set how many exposures per field we want.

In [ ]:
num_science_exp = 3
science_exptime = 1020

## Finalize the OB-xmls so they will be read correctly by the Observatory Control System (OCS). This step is essential, as the TL-OB-XML structure differs significantly from regular MOS OBs!!!

In [ ]:
from xml.parsers.expat import ExpatError

for xml in os.listdir(configure_output_path):
    if xml.endswith('.xml'):
        xml_path = os.path.join(configure_output_path, xml)
        try:
            tl_driver.finalize_tl_ob(xml_path, output_dir=final_configure_output_path,
                                   progtemp=progtemp,
                                   num_science_exposures=num_science_exp,
                                   science_exptime=science_exptime, 
                                   field_name='WTLS',
                                   xml_verbosity=True,
                                   drop_targinfo_from_field=True,
                                   # casuid_minus_1=False
                                   )

        # Checking for blank xmls due to a field position angle outside the rotator limits
        except ExpatError as e:
            print(f"Skipping {xml} due to XML parsing error: {e}")
            continue
        except SystemExit as e:
            print(f"Skipping {xml} due to SystemExit: {e} \n"
                  "Likely caused due to field position angle being outside the positioner limits, \n"
                  "resulting in an empty configure output xml.")
            continue 
        except AttributeError as e:
            print(f"Skipping {xml} due to AttributeError: {e}.")
            continue


## The final OB-XMLs can now be uploaded to WASP for validation. They can be found in the finalized_* folder inside the xmls/configure_* directory.
